In [11]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)

# Set style
plt.style.use('default')
sns.set_palette("husl")
%matplotlib inline

In [12]:
# Define dataset paths
dataset_dir = './datasets/'

# Load all datasets
datasets = {}
dataset_files = [
    'Botnet-Friday-02-03-2018_TrafficForML_CICFlowMeter.parquet',
    'Bruteforce-Wednesday-14-02-2018_TrafficForML_CICFlowMeter.parquet',
    'DDoS1-Tuesday-20-02-2018_TrafficForML_CICFlowMeter.parquet',
    'DDoS2-Wednesday-21-02-2018_TrafficForML_CICFlowMeter.parquet',
    'DoS1-Thursday-15-02-2018_TrafficForML_CICFlowMeter.parquet',
    'DoS2-Friday-16-02-2018_TrafficForML_CICFlowMeter.parquet',
    'Infil1-Wednesday-28-02-2018_TrafficForML_CICFlowMeter.parquet',
    'Infil2-Thursday-01-03-2018_TrafficForML_CICFlowMeter.parquet',
    'Web1-Thursday-22-02-2018_TrafficForML_CICFlowMeter.parquet',
    'Web2-Friday-23-02-2018_TrafficForML_CICFlowMeter.parquet'
]

# Extract labels from filenames
attack_labels = [f.split('-')[0] for f in dataset_files]

for label, file in zip(attack_labels, dataset_files):
    datasets[label] = pd.read_parquet(dataset_dir + file)
    print(f"Loaded {label}: {datasets[label].shape}")

print(f"\nTotal files loaded: {len(datasets)}")

Loaded Botnet: (771587, 78)
Loaded Bruteforce: (619346, 78)
Loaded DDoS1: (954846, 78)
Loaded DDoS2: (561396, 78)
Loaded DoS1: (794812, 78)
Loaded DoS2: (591873, 78)
Loaded Infil1: (456873, 78)
Loaded Infil2: (249170, 78)
Loaded Web1: (830224, 78)
Loaded Web2: (829405, 78)

Total files loaded: 10


## 2. Dataset Overview

In [13]:
# Create summary of dataset sizes
dataset_sizes = pd.DataFrame({
    'Attack Type': list(datasets.keys()),
    'Samples': [datasets[key].shape[0] for key in datasets.keys()],
    'Features': [datasets[key].shape[1] for key in datasets.keys()]
}).sort_values('Samples', ascending=False)

print(dataset_sizes.to_string(index=False))
print(f"\nTotal samples across all datasets: {dataset_sizes['Samples'].sum():,}")
print(f"Average samples per attack type: {dataset_sizes['Samples'].mean():.0f}")

Attack Type  Samples  Features
      DDoS1   954846        78
       Web1   830224        78
       Web2   829405        78
       DoS1   794812        78
     Botnet   771587        78
 Bruteforce   619346        78
       DoS2   591873        78
      DDoS2   561396        78
     Infil1   456873        78
     Infil2   249170        78

Total samples across all datasets: 6,659,532
Average samples per attack type: 665953


In [14]:
# Visualize dataset sizes
fig = px.bar(dataset_sizes.sort_values('Samples', ascending=True), 
             y='Attack Type', x='Samples',
             orientation='h',
             title='Sample Counts by Attack Type',
             labels={'Samples': 'Number of Samples'},
             color='Samples',
             color_continuous_scale='Viridis')
fig.show()

In [20]:

for i in list(datasets.values()):
    print(i.columns)
    print(i['Label'].value_counts())
    print("\n")

Index(['Protocol', 'Flow Duration', 'Total Fwd Packets',
       'Total Backward Packets', 'Fwd Packets Length Total',
       'Bwd Packets Length Total', 'Fwd Packet Length Max',
       'Fwd Packet Length Min', 'Fwd Packet Length Mean',
       'Fwd Packet Length Std', 'Bwd Packet Length Max',
       'Bwd Packet Length Min', 'Bwd Packet Length Mean',
       'Bwd Packet Length Std', 'Flow Bytes/s', 'Flow Packets/s',
       'Flow IAT Mean', 'Flow IAT Std', 'Flow IAT Max', 'Flow IAT Min',
       'Fwd IAT Total', 'Fwd IAT Mean', 'Fwd IAT Std', 'Fwd IAT Max',
       'Fwd IAT Min', 'Bwd IAT Total', 'Bwd IAT Mean', 'Bwd IAT Std',
       'Bwd IAT Max', 'Bwd IAT Min', 'Fwd PSH Flags', 'Bwd PSH Flags',
       'Fwd URG Flags', 'Bwd URG Flags', 'Fwd Header Length',
       'Bwd Header Length', 'Fwd Packets/s', 'Bwd Packets/s',
       'Packet Length Min', 'Packet Length Max', 'Packet Length Mean',
       'Packet Length Std', 'Packet Length Variance', 'FIN Flag Count',
       'SYN Flag Count', 'RST Fla

In [16]:
# Examine first dataset structure
first_dataset = list(datasets.values())[0]  
first_dataset.head()

,Protocol,Flow Duration,Total Fwd Packets,Total Backward Packets,Fwd Packets Length Total,Bwd Packets Length Total,Fwd Packet Length Max,Fwd Packet Length Min,Fwd Packet Length Mean,Fwd Packet Length Std,Bwd Packet Length Max,Bwd Packet Length Min,Bwd Packet Length Mean,Bwd Packet Length Std,Flow Bytes/s,Flow Packets/s,Flow IAT Mean,Flow IAT Std,Flow IAT Max,Flow IAT Min,Fwd IAT Total,Fwd IAT Mean,Fwd IAT Std,Fwd IAT Max,Fwd IAT Min,Bwd IAT Total,Bwd IAT Mean,Bwd IAT Std,Bwd IAT Max,Bwd IAT Min,Fwd PSH Flags,Bwd PSH Flags,Fwd URG Flags,Bwd URG Flags,Fwd Header Length,Bwd Header Length,Fwd Packets/s,Bwd Packets/s,Packet Length Min,Packet Length Max,Packet Length Mean,Packet Length Std,Packet Length Variance,FIN Flag Count,SYN Flag Count,RST Flag Count,PSH Flag Count,ACK Flag Count,URG Flag Count,CWE Flag Count,ECE Flag Count,Down/Up Ratio,Avg Packet Size,Avg Fwd Segment Size,Avg Bwd Segment Size,Fwd Avg Bytes/Bulk,Fwd Avg Packets/Bulk,Fwd Avg Bulk Rate,Bwd Avg Bytes/Bulk,Bwd Avg Packets/Bulk,Bwd Avg Bulk Rate,Subflow Fwd Packets,Subflow Fwd Bytes,Subflow Bwd Packets,Subflow Bwd Bytes,Init Fwd Win Bytes,Init Bwd Win Bytes,Fwd Act Data Packets,Fwd Seg Size Min,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label
0,0,112641719,3,0,0,0,0,0,0.000000,0.000000,0,0,0.000000,0.000000,0.000000,0.026633,5.632086e+07,139.300034,56320958,56320761,112641719,5.632086e+07,139.300034,56320958,56320761,0,0.0000,0.00000,0,0,0,0,0,0,0,0,0.026633,0.000000,0,0,0.00000,0.000000,0.000000,0,0,0,0,0,0,0,0,0,0.000000,0.000000,0.000000,0,0,0,0,0,0,3,0,0,0,-1,-1,0,0,0.0,0.0,0,0,56320860.0,139.300034,56320958,56320761,Benign
1,0,112641466,3,0,0,0,0,0,0.000000,0.000000,0,0,0.000000,0.000000,0.000000,0.026633,5.632073e+07,114.551300,56320814,56320652,112641466,5.632073e+07,114.551300,56320814,56320652,0,0.0000,0.00000,0,0,0,0,0,0,0,0,0.026633,0.000000,0,0,0.00000,0.000000,0.000000,0,0,0,0,0,0,0,0,0,0.000000,0.000000,0.000000,0,0,0,0,0,0,3,0,0,0,-1,-1,0,0,0.0,0.0,0,0,56320732.0,114.551300,56320814,56320652,Benign
2,0,112638623,3,0,0,0,0,0,0.000000,0.000000,0,0,0.000000,0.000000,0.000000,0.026634,5.631931e+07,301.934601,56319525,56319098,112638623,5.631931e+07,301.934601,56319525,56319098,0,0.0000,0.00000,0,0,0,0,0,0,0,0,0.026634,0.000000,0,0,0.00000,0.000000,0.000000,0,0,0,0,0,0,0,0,0,0.000000,0.000000,0.000000,0,0,0,0,0,0,3,0,0,0,-1,-1,0,0,0.0,0.0,0,0,56319312.0,301.934601,56319525,56319098,Benign
3,6,6453966,15,10,1239,2273,744,0,82.599998,196.741241,976,0,227.300003,371.677887,544.161528,3.873587,2.689152e+05,247443.781250,673900,22,6453966,4.609976e+05,123109.421875,673900,229740,5637902,626433.5625,455082.21875,1167293,554,0,0,0,0,488,328,2.324152,1.549435,0,976,135.07692,277.834747,77192.156250,0,0,0,1,0,0,0,0,0,140.479996,82.599998,227.300003,0,0,0,0,0,0,15,1239,10,2273,65535,233,6,32,0.0,0.0,0,0,0.0,0.000000,0,0,Benign
4,6,8804066,14,11,1143,2209,744,0,81.642860,203.745544,976,0,200.818176,362.249878,380.733175,2.839597,3.668361e+05,511356.625000,1928102,21,8804066,6.772359e+05,532417.000000,1928102,246924,7715481,771548.1250,755543.06250,2174893,90,0,0,0,0,456,360,1.590174,1.249423,0,976,128.92308,279.763031,78267.351562,0,0,0,1,0,0,0,0,0,134.080002,81.642860,200.818176,0,0,0,0,0,0,14,1143,11,2209,5808,233,6,32,0.0,0.0,0,0,0.0,0.000000,0,0,Benign


In [9]:

print(f"\nData types:\n{first_dataset.dtypes}")
print(f"\nMissing values:\n{first_dataset.isnull().sum()[first_dataset.isnull().sum() > 0]}")


Data types:
Protocol                        int8
Flow Duration                  int32
Total Fwd Packets              int32
Total Backward Packets         int32
Fwd Packets Length Total       int32
                              ...   
Idle Mean                    float32
Idle Std                     float32
Idle Max                     float32
Idle Min                     float32
Label                       category
Length: 78, dtype: object

Missing values:
Series([], dtype: int64)


## 3. Feature Analysis

In [6]:
# Get numeric columns
numeric_cols = first_dataset.select_dtypes(include=[np.number]).columns.tolist()
print(f"Number of numeric features: {len(numeric_cols)}")
print(f"\nNumeric features:\n{numeric_cols}")

Number of numeric features: 77

Numeric features:
['Protocol', 'Flow Duration', 'Total Fwd Packets', 'Total Backward Packets', 'Fwd Packets Length Total', 'Bwd Packets Length Total', 'Fwd Packet Length Max', 'Fwd Packet Length Min', 'Fwd Packet Length Mean', 'Fwd Packet Length Std', 'Bwd Packet Length Max', 'Bwd Packet Length Min', 'Bwd Packet Length Mean', 'Bwd Packet Length Std', 'Flow Bytes/s', 'Flow Packets/s', 'Flow IAT Mean', 'Flow IAT Std', 'Flow IAT Max', 'Flow IAT Min', 'Fwd IAT Total', 'Fwd IAT Mean', 'Fwd IAT Std', 'Fwd IAT Max', 'Fwd IAT Min', 'Bwd IAT Total', 'Bwd IAT Mean', 'Bwd IAT Std', 'Bwd IAT Max', 'Bwd IAT Min', 'Fwd PSH Flags', 'Bwd PSH Flags', 'Fwd URG Flags', 'Bwd URG Flags', 'Fwd Header Length', 'Bwd Header Length', 'Fwd Packets/s', 'Bwd Packets/s', 'Packet Length Min', 'Packet Length Max', 'Packet Length Mean', 'Packet Length Std', 'Packet Length Variance', 'FIN Flag Count', 'SYN Flag Count', 'RST Flag Count', 'PSH Flag Count', 'ACK Flag Count', 'URG Flag Count

In [ ]:
# Statistical summary of numeric features for first dataset
print("Statistical Summary (first dataset):")
print(first_dataset[numeric_cols].describe())

In [ ]:
# Combine all data for comprehensive analysis
# Add labels to identify attack types
dfs_with_labels = []
for label, df in datasets.items():
    df_copy = df.copy()
    df_copy['Attack_Type'] = label
    dfs_with_labels.append(df_copy)

combined_df = pd.concat(dfs_with_labels, ignore_index=True)
print(f"Combined dataset shape: {combined_df.shape}")
print(f"\nAttack type distribution:")
print(combined_df['Attack_Type'].value_counts())

## 4. Feature Distributions

In [ ]:
# Select top features by variance for visualization
feature_variance = combined_df[numeric_cols].var().sort_values(ascending=False)
top_features = feature_variance.head(10).index.tolist()

print("Top 10 features by variance:")
print(feature_variance.head(10))

In [ ]:
# Distribution plots for top features
n_features = len(top_features)
fig, axes = plt.subplots((n_features + 2) // 3, 3, figsize=(15, 12))
axes = axes.flatten()

for idx, feature in enumerate(top_features):
    axes[idx].hist(combined_df[feature].dropna(), bins=50, alpha=0.7, color='steelblue', edgecolor='black')
    axes[idx].set_title(f'{feature}', fontsize=10, fontweight='bold')
    axes[idx].set_xlabel('Value')
    axes[idx].set_ylabel('Frequency')
    axes[idx].grid(True, alpha=0.3)

# Hide extra subplots
for idx in range(n_features, len(axes)):
    axes[idx].set_visible(False)

plt.tight_layout()
plt.show()

In [ ]:
# Box plots by attack type for top features
fig = make_subplots(rows=2, cols=2, subplot_titles=top_features[:4])

for idx, feature in enumerate(top_features[:4]):
    row = idx // 2 + 1
    col = idx % 2 + 1
    
    for attack_type in combined_df['Attack_Type'].unique():
        data = combined_df[combined_df['Attack_Type'] == attack_type][feature].dropna()
        fig.add_trace(
            go.Box(y=data, name=attack_type, boxmean='sd'),
            row=row, col=col
        )

fig.update_layout(height=800, title_text="Feature Distributions by Attack Type (Top 4)")
fig.show()

## 5. Missing Values Analysis

In [ ]:
# Check missing values per dataset
missing_summary = []
for label, df in datasets.items():
    missing_count = df.isnull().sum()
    missing_pct = (missing_count / len(df) * 100)
    missing_summary.append({
        'Attack_Type': label,
        'Total_Missing': missing_count.sum(),
        'Missing_Features': (missing_count > 0).sum(),
        'Max_Missing_%': missing_pct.max() if missing_pct.max() > 0 else 0
    })

missing_df = pd.DataFrame(missing_summary).sort_values('Total_Missing', ascending=False)
print(missing_df.to_string(index=False))

In [ ]:
# Visualize missing values
missing_data = combined_df[numeric_cols].isnull().sum()
missing_data = missing_data[missing_data > 0].sort_values(ascending=False)

if len(missing_data) > 0:
    fig = px.bar(x=missing_data.index, y=missing_data.values,
                 title='Missing Values by Feature',
                 labels={'x': 'Features', 'y': 'Count'},
                 color=missing_data.values,
                 color_continuous_scale='Reds')
    fig.show()
else:
    print("No missing values found in numeric features.")

## 6. Correlation Analysis

In [ ]:
# Calculate correlation matrix for combined data
correlation_matrix = combined_df[numeric_cols].corr()

# Find highly correlated features
high_corr_pairs = []
for i in range(len(correlation_matrix.columns)):
    for j in range(i+1, len(correlation_matrix.columns)):
        if abs(correlation_matrix.iloc[i, j]) > 0.95:
            high_corr_pairs.append({
                'Feature_1': correlation_matrix.columns[i],
                'Feature_2': correlation_matrix.columns[j],
                'Correlation': correlation_matrix.iloc[i, j]
            })

if high_corr_pairs:
    high_corr_df = pd.DataFrame(high_corr_pairs).sort_values('Correlation', key=abs, ascending=False)
    print(f"Highly correlated feature pairs (|r| > 0.95): {len(high_corr_df)}")
    print(high_corr_df.head(10).to_string(index=False))
else:
    print("No highly correlated feature pairs found.")

In [ ]:
# Correlation heatmap for top features
top_corr_features = feature_variance.head(15).index.tolist()
corr_subset = combined_df[top_corr_features].corr()

fig = px.imshow(corr_subset,
                labels=dict(color='Correlation'),
                title='Correlation Matrix: Top 15 Features by Variance',
                color_continuous_scale='RdBu_r',
                zmin=-1, zmax=1,
                height=700, width=700)
fig.show()

## 7. Outlier Detection

In [ ]:
# Calculate outliers using IQR method
def find_outliers_iqr(data):
    Q1 = data.quantile(0.25)
    Q3 = data.quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    return ((data < lower_bound) | (data > upper_bound)).sum()

outlier_summary = {}
for feature in top_features:
    outlier_count = find_outliers_iqr(combined_df[feature].dropna())
    outlier_pct = (outlier_count / len(combined_df)) * 100
    outlier_summary[feature] = {'count': outlier_count, 'percentage': outlier_pct}

outlier_df = pd.DataFrame(outlier_summary).T.sort_values('count', ascending=False)
print("Outliers by Feature (IQR method):")
print(outlier_df.head(10))

## 8. Attack Type Characteristics

In [ ]:
# Mean feature values by attack type
attack_means = combined_df.groupby('Attack_Type')[numeric_cols].mean()

# Normalize for better visualization
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
attack_means_normalized = pd.DataFrame(
    scaler.fit_transform(attack_means[top_features]),
    columns=top_features,
    index=attack_means.index
)

fig = px.imshow(attack_means_normalized,
                labels=dict(color='Normalized Mean Value'),
                title='Feature Means by Attack Type (Top 10 Features, Normalized)',
                color_continuous_scale='RdYlGn',
                height=600, width=900)
fig.show()

In [ ]:
# Radar chart for attack type profiles
from math import pi

top_5_features = feature_variance.head(5).index.tolist()

fig = go.Figure()

for attack_type in combined_df['Attack_Type'].unique():
    attack_data = combined_df[combined_df['Attack_Type'] == attack_type][top_5_features].mean()
    attack_data_normalized = (attack_data - combined_df[top_5_features].mean()) / combined_df[top_5_features].std()
    
    fig.add_trace(go.Scatterpolar(
        r=attack_data_normalized.values,
        theta=top_5_features,
        fill='toself',
        name=attack_type
    ))

fig.update_layout(
    polar=dict(radialaxis=dict(visible=True, range=[-3, 3])),
    title='Attack Type Profiles (Normalized by Top 5 Features)',
    height=700, width=900
)
fig.show()

## 9. Data Quality Metrics

In [ ]:
# Calculate data quality metrics
quality_metrics = {}

for label, df in datasets.items():
    numeric_data = df.select_dtypes(include=[np.number])
    
    # Completeness
    completeness = (1 - numeric_data.isnull().sum().sum() / (len(numeric_data) * len(numeric_data.columns))) * 100
    
    # Duplicates
    duplicates = df.duplicated().sum()
    duplicate_pct = (duplicates / len(df)) * 100
    
    # Infinite values
    inf_count = np.isinf(numeric_data).sum().sum()
    
    quality_metrics[label] = {
        'Completeness_%': completeness,
        'Duplicates': duplicates,
        'Duplicate_%': duplicate_pct,
        'Infinite_Values': inf_count
    }

quality_df = pd.DataFrame(quality_metrics).T
print(quality_df.round(2))

In [ ]:
# Visualize completeness
fig = px.bar(quality_df.reset_index().rename(columns={'index': 'Attack_Type'}), 
             x='Attack_Type', y='Completeness_%',
             title='Data Completeness by Attack Type',
             color='Completeness_%',
             color_continuous_scale='Greens',
             labels={'Completeness_%': 'Completeness (%)'})
fig.show()

## 10. Summary and Key Insights

In [ ]:
print("="*70)
print("DATASET SUMMARY")
print("="*70)
print(f"\n1. DATASET OVERVIEW:")
print(f"   - Total samples: {len(combined_df):,}")
print(f"   - Total features: {len(numeric_cols)}")
print(f"   - Attack types: {combined_df['Attack_Type'].nunique()}")
print(f"   - Largest dataset: {dataset_sizes.iloc[0]['Attack Type']} ({dataset_sizes.iloc[0]['Samples']:,} samples)")
print(f"   - Smallest dataset: {dataset_sizes.iloc[-1]['Attack Type']} ({dataset_sizes.iloc[-1]['Samples']:,} samples)")

print(f"\n2. DATA QUALITY:")
print(f"   - Average completeness: {quality_df['Completeness_%'].mean():.2f}%")
print(f"   - Total duplicate rows: {combined_df.duplicated().sum():,}")
print(f"   - Missing values: {combined_df[numeric_cols].isnull().sum().sum()}")

print(f"\n3. FEATURE STATISTICS:")
print(f"   - Features with high variance (top 3):")
for feat in feature_variance.head(3).index:
    print(f"     • {feat}: {feature_variance[feat]:.2e}")

if len(high_corr_pairs) > 0:
    print(f"\n4. CORRELATION INSIGHTS:")
    print(f"   - Highly correlated pairs found: {len(high_corr_pairs)}")
    print(f"   - Strongest correlation: {abs(high_corr_pairs[0]['Correlation']):.4f}")
else:
    print(f"\n4. CORRELATION INSIGHTS:")
    print(f"   - No highly correlated pairs (|r| > 0.95)")

print(f"\n5. OUTLIERS:")
total_outliers = outlier_df['count'].sum()
print(f"   - Total outlier instances detected: {total_outliers:,}")
print(f"   - Feature with most outliers: {outlier_df['count'].idxmax()} ({int(outlier_df['count'].max())} instances)")

print("\n" + "="*70)

## Next Steps

1. **Data Preprocessing**: Handle missing values, outliers, and feature scaling
2. **Feature Engineering**: Create derived features from high-variance attributes
3. **Class Balance**: Analyze and address any imbalance in attack type distribution
4. **Feature Selection**: Use correlation and variance analysis for feature selection
5. **Model Development**: Train classifiers with the cleaned and engineered features